In [1]:


import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
import time
from typing import List, Dict, Optional
from dataclasses import dataclass
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin, urlparse
import re

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)



In [2]:
@dataclass
class Article:
    """Data class for article information"""
    title: str
    content: str
    url: str
    source: str
    scores: Optional[Dict[str, float]] = None
    
    def to_dict(self):
        return {
            'title': self.title,
            'content': self.content[:500],  # Preview only
            'url': self.url,
            'source': self.source,
            **({f'score_{k}': v for k, v in self.scores.items()} if self.scores else {})
        }


class ArticleScraper:
    """
    Flexible web scraper for news and research articles
    """
    
    def __init__(self, timeout=10, max_retries=3):
        self.timeout = timeout
        self.max_retries = max_retries
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
    
    def scrape_nature_smart(self, max_articles):
        """
        Start from Nature's subjects page and follow the link hierarchy:
        subjects page → individual subject pages → article listing pages → articles
        
        Works with: https://www.nature.com/subjects
        """
        from collections import deque
        
        article_links = set()
        visited_pages = set()
        subject_pages = set()
        
        # Level 1: Start from subjects index
        subjects_index = "https://www.nature.com/subjects"
        
        print(f"Step 1: Fetching subject categories from {subjects_index}")
        
        try:
            response = requests.get(subjects_index, timeout=10, headers=self.headers)
            if response.status_code != 200:
                print(f"Failed to load subjects page: {response.status_code}")
                return []
        except requests.RequestException as e:
            print(f"Error fetching subjects page: {e}")
            return []
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Find all subject links (e.g., /subjects/biochemistry, /subjects/genetics)
        for link in soup.find_all("a", href=True):
            href = link.get('href')
            if not href:
                continue
            
            full_url = urljoin(subjects_index, href)
            
            # Look for subject pages like /subjects/biochemistry
            if "/subjects/" in full_url and full_url != subjects_index:
                # Clean URL
                parsed = urlparse(full_url)
                clean_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
                subject_pages.add(clean_url)
        
        print(f"Step 2: Found {len(subject_pages)} subject categories")
        for i, subject in enumerate(list(subject_pages)[:5], 1):
            print(f"  {i}. {subject}")
        if len(subject_pages) > 5:
            print(f"  ... and {len(subject_pages) - 5} more")
        
        # Level 2: Visit each subject page and collect articles with pagination
        for subject_url in subject_pages:
            if len(article_links) >= max_articles:
                break
            
            print(f"\nStep 3: Exploring subject - {subject_url}")
            
            # Try pagination on each subject page
            for page_num in range(1, 31):  # Up to 30 pages per subject
                if len(article_links) >= max_articles:
                    break
                
                # Add pagination
                if "?" in subject_url:
                    page_url = f"{subject_url}&page={page_num}"
                else:
                    page_url = f"{subject_url}?page={page_num}"
                
                if page_url in visited_pages:
                    continue
                
                visited_pages.add(page_url)
                
                if page_num == 1:
                    print(f"  Checking pages...")
                
                try:
                    response = requests.get(page_url, timeout=10, headers=self.headers)
                    if response.status_code != 200:
                        if page_num == 1:
                            print(f"  Failed to load (status {response.status_code})")
                        break
                except requests.RequestException as e:
                    if page_num == 1:
                        print(f"  Error: {e}")
                    break
                
                soup = BeautifulSoup(response.text, "html.parser")
                
                articles_on_page = 0
                
                # Find article links
                for link in soup.find_all("a", href=True):
                    href = link.get('href')
                    if not href:
                        continue
                    
                    full_url = urljoin(page_url, href)
                    
                    # Check if it's an article
                    if "/articles/" in full_url and not full_url.endswith(".pdf"):
                        if full_url not in article_links:
                            article_links.add(full_url)
                            articles_on_page += 1
                            
                            if len(article_links) % 10 == 0:
                                print(f"  ✓ {len(article_links)} articles found so far...")
                            
                            if len(article_links) >= max_articles:
                                break
                
                # Stop if no articles found on this page
                if articles_on_page == 0 and page_num > 1:
                    break
                
                time.sleep(0.5)
        
        print(f"\n{'='*80}")
        print(f"COMPLETE: Found {len(article_links)} articles from {len(subject_pages)} subjects")
        print(f"{'='*80}")
        
        return list(article_links)
    
    def scrape_nature_articles_diagnostic(self, max_articles=200):
        """
        Diagnostic version - see exactly what's happening
        """
        article_links = set()
        
        starting_urls = [
            "https://www.nature.com/nature/articles",
            "https://www.nature.com/search?article_type=protocols,research,reviews&subject=biochemistry",
            "https://www.nature.com/subjects/genetics",
        ]
        
        for base_url in starting_urls:
            print(f"\n{'='*80}")
            print(f"Checking section: {base_url}")
            print('='*80)
            
            # Check just first page to see what's there
            try:
                response = requests.get(base_url, timeout=10, headers=self.headers)
                print(f"Status: {response.status_code}")
            except requests.RequestException as e:
                print(f"Error: {e}")
                continue
            
            soup = BeautifulSoup(response.text, "html.parser")
            
            # Count ALL links
            all_links = soup.find_all("a", href=True)
            print(f"Total <a> tags found: {len(all_links)}")
            
            # Count article links
            article_count = 0
            sample_articles = []
            for link in all_links:
                href = link.get('href')
                if href and "/articles/" in href:
                    article_count += 1
                    full_url = urljoin(base_url, href)
                    if len(sample_articles) < 5:
                        sample_articles.append(full_url)
            
            print(f"Links with '/articles/': {article_count}")
            print(f"\nSample articles found:")
            for i, url in enumerate(sample_articles, 1):
                print(f"  {i}. {url}")
            
            time.sleep(1)
        
        return []

    def scrape_url(self, url: str) -> Optional[Article]:
        """
        Scrape a single article URL
        
        Returns Article object or None if scraping fails
        """
        for attempt in range(self.max_retries):
            try:
                response = requests.get(url, headers=self.headers, timeout=self.timeout)
                response.raise_for_status()
                
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Extract title and content using multiple strategies
                title = self._extract_title(soup)
                content = self._extract_content(soup)
                
                if not title or not content:
                    logger.warning(f"Could not extract title or content from {url}")
                    return None
                
                source = urlparse(url).netloc
                
                return Article(
                    title=title,
                    content=content,
                    url=url,
                    source=source
                )
                
            except requests.RequestException as e:
                logger.error(f"Attempt {attempt + 1}/{self.max_retries} failed for {url}: {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                continue
        
        return None
    
    def _extract_title(self, soup: BeautifulSoup) -> str:
        """Extract article title using multiple strategies"""
        # Strategy 1: OpenGraph meta tag
        og_title = soup.find('meta', property='og:title')
        if og_title and og_title.get('content'):
            return og_title['content'].strip()
        
        # Strategy 2: Standard title tag
        if soup.title:
            return soup.title.string.strip()
        
        # Strategy 3: h1 tag
        h1 = soup.find('h1')
        if h1:
            return h1.get_text(strip=True)
        
        return ""
    
    def _extract_content(self, soup: BeautifulSoup) -> str:
        """Extract article content using multiple strategies"""
        # Strategy 1: OpenGraph description
        og_desc = soup.find('meta', property='og:description')
        if og_desc and og_desc.get('content'):
            content = og_desc['content'].strip()
        else:
            content = ""
        
        # Strategy 2: Article tag
        article = soup.find('article')
        if article:
            paragraphs = article.find_all('p')
            content += ' ' + ' '.join([p.get_text(strip=True) for p in paragraphs])
        
        # Strategy 3: Main content div (common patterns)
        if not content:
            main_content = soup.find(['div'], class_=re.compile(r'(article|content|post|entry|body)', re.I))
            if main_content:
                paragraphs = main_content.find_all('p')
                content = ' '.join([p.get_text(strip=True) for p in paragraphs])
        
        # Strategy 4: All paragraphs as fallback
        if not content or len(content) < 100:
            paragraphs = soup.find_all('p')
            content = ' '.join([p.get_text(strip=True) for p in paragraphs[:10]])
        
        # Clean content
        content = re.sub(r'\s+', ' ', content).strip()
        
        return content
    
    def scrape_arxiv(self, arxiv_id: str) -> Optional[Article]:
        """
        Specialized scraper for arXiv research papers
        
        Args:
            arxiv_id: arXiv ID (e.g., "2301.12345")
        """
        url = f"https://arxiv.org/abs/{arxiv_id}"
        domain = urlparse(url).netloc.lower()
        try:
            if "arxiv.org" in domain:
                m = re.search(r"/abs/([^/?#]+)", url)
                if m:
                    return self.scrape_arxiv(m.group(1))
                return None
        
            # PubMed (use API handler - add this function)
            if "pubmed.ncbi.nlm.nih.gov" in domain:
                m = re.search(r"pubmed\.ncbi\.nlm\.nih\.gov/(\d+)", url)
                if m:
                    return self.scrape_pubmed(m.group(1))
                return None
        
        # OpenAlex pages are NOT articles
            if "openalex.org" in domain:
                return None
            response = requests.get(url, headers=self.headers, timeout=self.timeout)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Title
            title_tag = soup.find('h1', class_='title')
            title = title_tag.get_text(strip=True).replace('Title:', '').strip() if title_tag else ""
            
            # Abstract
            abstract_tag = soup.find('blockquote', class_='abstract')
            abstract = abstract_tag.get_text(strip=True).replace('Abstract:', '').strip() if abstract_tag else ""
            
            return Article(
                title=title,
                content=abstract,
                url=url,
                source='arxiv.org'
            )
            
        except Exception as e:
            logger.error(f"Failed to scrape arXiv {arxiv_id}: {e}")
            return None
    
    def scrape_multiple(self, urls: List[str], max_workers=5) -> List[Article]:
        """
        Scrape multiple URLs concurrently
        
        Args:
            urls: List of article URLs
            max_workers: Number of concurrent threads
        """
        articles = []
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_url = {executor.submit(self.scrape_url, url): url for url in urls}
            
            for future in as_completed(future_to_url):
                url = future_to_url[future]
                try:
                    article = future.result()
                    if article:
                        articles.append(article)
                except Exception as e:
                    logger.error(f"Error processing {url}: {e}")
        
        return articles

In [8]:
class MultiDimensionalAnalyzer:
    """
    Analyzes articles across 5 dimensions
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.embedding_model = SentenceTransformer(model_name)
        self.regression_model = MultiOutputRegressor(
            RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        )
        self.is_fitted = False
        
        self.dimensions = [
            'growth_potential',
            'recession_resistance',
            'automation_resistance',
            'skill_accessibility',
            'cross_industry_collaboration'
        ]
    
    def train(self, texts: List[str], scores_df: pd.DataFrame):
        """Train the model on labeled data"""
        logger.info("Creating embeddings for training data...")
        embeddings = self.embedding_model.encode(texts, batch_size=32, show_progress_bar=False)
        
        scores_array = scores_df[self.dimensions].values
        
        logger.info("Training multi-output model...")
        self.regression_model.fit(embeddings, scores_array)
        self.is_fitted = True
        
        logger.info("Training complete!")
        return self
    
    def score_articles(self, articles: List[Article]) -> List[Article]:
        """
        Score multiple articles across all dimensions
        
        Args:
            articles: List of Article objects
            
        Returns:
            Same articles with scores populated
        """
        if not self.is_fitted:
            raise ValueError("Model must be trained first!")
        
        # Combine title and content for better context
        texts = [f"{art.title}. {art.content}" for art in articles]
        
        logger.info(f"Scoring {len(articles)} articles...")
        embeddings = self.embedding_model.encode(texts, batch_size=32, show_progress_bar=False)
        predictions = self.regression_model.predict(embeddings)
        predictions = np.clip(predictions, 0, 10)
        
        # Add scores to articles
        for i, article in enumerate(articles):
            article.scores = {
                dim: float(predictions[i, j]) 
                for j, dim in enumerate(self.dimensions)
            }
        
        return articles


class ArticlePipeline:
    """
    Complete pipeline: Scrape -> Analyze -> Store -> Recommend Majors
    """
    
    def __init__(self, analyzer: MultiDimensionalAnalyzer, scraper: Optional[ArticleScraper] = None):
        self.analyzer = analyzer
        self.scraper = scraper or ArticleScraper()
        self.vectorizedArticles: List[Dict] = []  # Main storage
        self.major_mapping = self._create_major_mapping()  # Map topics to majors
    
    def save_complete(self, articles_file='articles.json', model_file='analyzer_model.pkl'):
        """
        Save both articles and trained model
        """
        import pickle
        import json
        
        # Save articles as JSON
        with open(articles_file, 'w', encoding='utf-8') as f:
            json.dump(self.vectorizedArticles, f, indent=2, ensure_ascii=False)
        
        # Save the trained analyzer (includes the regression model)
        with open(model_file, 'wb') as f:
            pickle.dump(self.analyzer, f)
        
        print(f"✓ Saved articles to {articles_file}")
        print(f"✓ Saved trained model to {model_file}")

    def load_complete(self, articles_file='articles.json', model_file='analyzer_model.pkl'):
        """
        Load both articles and trained model
        """
        import pickle
        import json
        
        # Load articles
        try:
            with open(articles_file, 'r', encoding='utf-8') as f:
                self.vectorizedArticles = json.load(f)
            print(f"✓ Loaded {len(self.vectorizedArticles)} articles")
        except FileNotFoundError:
            print(f"Articles file {articles_file} not found")
            return False
        
        # Load model
        try:
            with open(model_file, 'rb') as f:
                self.analyzer = pickle.load(f)
            print(f"✓ Loaded trained model")
            return True
        except FileNotFoundError:
            print(f"Model file {model_file} not found")
            return False

    def _create_major_mapping(self) -> Dict[str, List[str]]:
        """
        Map article topics/keywords to relevant college majors
        Customize this based on your needs
        
        IMPORTANT: Order matters! More specific majors should come FIRST
        to prevent everything being classified as CS/general categories
        """
        return {
            # STEM - Specific first
            'biomedical_engineering': ['biomedical engineering', 'medical device', 'prosthetic', 
                                      'medical imaging', 'biosensor', 'tissue engineering',
                                      'medical technology', 'healthcare engineering'],
            'biology': ['crispr', 'gene therapy', 'genetics research', 'genomics', 
                       'proteomics', 'microbiology', 'molecular biology', 'cell biology',
                       'evolutionary biology', 'ecology', 'bioinformatics'],
            'chemistry': ['drug discovery', 'pharmaceutical chemistry', 'organic chemistry',
                         'chemical synthesis', 'catalyst', 'polymer chemistry', 'biochemistry',
                         'analytical chemistry', 'chemical engineering materials'],
            'neuroscience': ['neuroscience', 'brain research', 'cognitive neuroscience', 
                           'neuroimaging', 'brain mapping', 'neuroplasticity',
                           'neurotechnology', 'brain-computer interface', 'neural circuits'],
            'physics': ['particle physics', 'quantum mechanics', 'photonics', 'optics', 
                       'laser physics', 'condensed matter', 'astrophysics', 'cosmology',
                       'theoretical physics', 'quantum field'],
            'environmental_science': ['climate science', 'renewable energy technology', 
                                     'solar energy', 'wind power', 'carbon capture',
                                     'ecology', 'conservation', 'environmental policy',
                                     'climate modeling', 'oceanography'],
            'mechanical_engineering': ['mechanical design', 'automotive engineering', 
                                      'aerospace engineering', 'thermodynamics', 
                                      '3d printing technology', 'manufacturing systems',
                                      'fluid mechanics', 'structural engineering'],
            'electrical_engineering': ['semiconductor manufacturing', 'circuit design', 
                                      'power systems', 'signal processing', 'embedded systems',
                                      'microelectronics', 'vlsi design', 'electrical power'],
            'civil_engineering': ['infrastructure', 'structural design', 'urban planning',
                                 'transportation systems', 'water resources', 'construction',
                                 'geotechnical engineering', 'sustainable building'],
            
            # Computing - More specific categories
            'artificial_intelligence': ['machine learning research', 'deep learning architecture',
                                       'neural network design', 'natural language processing',
                                       'computer vision research', 'reinforcement learning',
                                       'ai safety', 'llm', 'transformer architecture'],
            'cybersecurity': ['cybersecurity', 'network security', 'cryptography implementation',
                            'penetration testing', 'security vulnerabilities', 
                            'information security', 'cyber defense'],
            'data_science': ['data analytics', 'statistical modeling', 'data mining techniques',
                           'predictive modeling', 'business analytics', 'data visualization'],
            'computer_science': ['software development', 'programming language design',
                                'operating systems', 'compiler design', 'distributed systems',
                                'computer architecture'],  # Most general CS last
            
            # Math
            'mathematics': ['pure mathematics', 'mathematical proof', 'number theory',
                          'topology', 'abstract algebra', 'mathematical analysis',
                          'combinatorics', 'game theory'],
            'applied_mathematics': ['mathematical modeling', 'optimization theory',
                                   'computational mathematics', 'numerical analysis',
                                   'operations research'],
            
            # Social Sciences & Humanities
            'psychology': ['psychological research', 'behavioral science', 'mental health research',
                          'developmental psychology', 'social psychology', 'clinical psychology',
                          'cognitive psychology'],
            'economics': ['economic theory', 'macroeconomics', 'microeconomics', 
                         'econometric analysis', 'market dynamics', 'fiscal policy',
                         'monetary policy', 'labor economics'],
            'political_science': ['political theory', 'international relations', 
                                'public policy analysis', 'governance systems',
                                'political institutions', 'diplomacy'],
            'sociology': ['social research', 'social inequality', 'urban sociology',
                         'sociological theory', 'demographic research', 'community studies'],
            
            # Professional/Applied
            'public_health': ['public health', 'epidemiology', 'health policy',
                            'disease prevention', 'health systems', 'global health'],
            'business': ['business strategy', 'entrepreneurship', 'venture capital',
                        'startup funding', 'business model innovation', 'corporate strategy'],
            'finance': ['financial markets', 'investment analysis', 'portfolio management',
                       'risk management', 'trading strategies', 'financial modeling'],
            'marketing': ['consumer behavior', 'brand strategy', 'digital marketing',
                         'market research', 'advertising'],
            
            # Interdisciplinary
            'cognitive_science': ['cognitive science', 'human cognition', 'decision making',
                                'perception research', 'cognitive modeling'],
            'materials_science': ['materials science', 'nanomaterials', 'advanced materials',
                                'material properties', 'metamaterials']
        }
    
    def process_urls(self, urls: List[str], max_workers=5) -> pd.DataFrame:
        """
        Complete pipeline: scrape URLs, analyze, and store
        
        Args:
            urls: List of article URLs to scrape
            max_workers: Concurrent scraping threads
            
        Returns:
            DataFrame with results
        """
        logger.info(f"Starting pipeline for {len(urls)} URLs...")
        
        # Step 1: Scrape articles
        articles = self.scraper.scrape_multiple(urls, max_workers=max_workers)
        logger.info(f"Successfully scraped {len(articles)}/{len(urls)} articles")
        
        if not articles:
            logger.warning("No articles scraped successfully!")
            return pd.DataFrame()
        
        # Step 2: Score articles
        scored_articles = self.analyzer.score_articles(articles)
        
        # Step 3: Store in vectorizedArticles
        for article in scored_articles:
            self.vectorizedArticles.append({
                'title': article.title,
                'url': article.url,
                'source': article.source,
                **article.scores
            })
        
        # Return as DataFrame for easy viewing
        df = pd.DataFrame(self.vectorizedArticles)
        df.to_csv('Vectorized Articles CSV')
        logger.info(f"Pipeline complete! Total articles in vectorizedArticles: {len(self.vectorizedArticles)}")

    def process_arxiv_papers(self, arxiv_ids: List[str]) -> pd.DataFrame:
        """Process arXiv papers specifically"""
        logger.info(f"Processing {len(arxiv_ids)} arXiv papers...")
        
        articles = []
        for arxiv_id in arxiv_ids:
            article = self.scraper.scrape_arxiv(arxiv_id)
            if article:
                articles.append(article)
            time.sleep(1)  # Respect rate limits
        
        logger.info(f"Successfully scraped {len(articles)}/{len(arxiv_ids)} papers")
        
        if articles:
            scored_articles = self.analyzer.score_articles(articles)
            
            for article in scored_articles:
                self.vectorizedArticles.append({
                    'title': article.title,
                    'url': article.url,
                    'source': article.source,
                    **article.scores
                })
        
        return pd.DataFrame(self.vectorizedArticles)
    
    def export_results(self, filename='article_scores.csv'):
        """Export vectorizedArticles to CSV"""
        df = pd.DataFrame(self.vectorizedArticles)
        df.to_csv(filename, index=False)
        logger.info(f"Exported {len(self.vectorizedArticles)} articles to {filename}")
        return df
    
    def get_top_articles(self, dimension: str, n=10) -> pd.DataFrame:
        """Get top N articles for a specific dimension"""
        df = pd.DataFrame(self.vectorizedArticles)
        return df.nlargest(n, dimension)[['title', 'source', dimension]]
    
    def _extract_article_topics(self, article: Dict, threshold: float = 0.1) -> List[str]:
        """
        Extract relevant majors using SEMANTIC matching instead of keywords
        Much more flexible - will catch related concepts even without exact keywords
        
        Args:
            article: Article dictionary with 'title' and 'content'
            threshold: Similarity threshold (0-1, default 0.1)
        """
        
        article_text = str(article.get('title', '')) + ' ' + str(article.get('content', ''))
        if not article_text.strip():
            return []

        # Create embedding for article
        article_embedding = self.analyzer.embedding_model.encode(
            [article_text], 
            show_progress_bar=False
        )[0]
        
        # Create embeddings for each major's description
        major_scores = {}
        
        for major, keywords in self.major_mapping.items():
            # Combine all keywords into a major description
            major_description = ' '.join(keywords)
            major_embedding = self.analyzer.embedding_model.encode(
                [major_description],
                show_progress_bar=False
            )[0]
            
            # Calculate cosine similarity
            similarity = np.dot(article_embedding, major_embedding) / (
                np.linalg.norm(article_embedding) * np.linalg.norm(major_embedding)
            )
            
            if similarity >= threshold:
                major_scores[major] = similarity
        
        # Return top 2 most similar majors
        if major_scores:
            sorted_majors = sorted(major_scores.items(), key=lambda x: x[1], reverse=True)
            
            if len(sorted_majors) == 1:
                return [sorted_majors[0][0]]
            elif len(sorted_majors) >= 2:
                # Include second major if it's at least 70% as similar as first
                if sorted_majors[1][1] >= sorted_majors[0][1] * 0.7:
                    return [sorted_majors[0][0], sorted_majors[1][0]]
                else:
                    return [sorted_majors[0][0]]
        
        return []
  
    def recommend_major(self, 
                       student_interests: List[str],
                       student_priorities: Dict[str, float] = None,
                       top_n: int = 5,
                       use_semantic_matching: bool = True,
                       semantic_threshold: float = 0.1) -> pd.DataFrame:
        """
        Recommend majors for a student based on their interests and priorities
        
        Args:
            student_interests: List of topics/keywords OR characteristics
                              Topics: ['artificial intelligence', 'healthcare', 'robotics']
                              Characteristics: ['research focused', 'abstract thinking', 'human-element']
                              Mix: ['AI', 'helping people', 'creative problem solving']
            student_priorities: Dictionary mapping dimension to importance weight (0-1)
                               e.g., {'growth_potential': 1.0, 'recession_resistance': 0.7, ...}
                               If None, all dimensions weighted equally
            top_n: Number of major recommendations to return
            use_semantic_matching: If True, uses AI embeddings for matching (better for characteristics)
                                   If False, uses simple keyword matching (faster but less flexible)
            semantic_threshold: Similarity threshold for semantic matching (0-1, default 0.3)
        
        Returns:
            DataFrame with recommended majors, their scores, and supporting articles
        """
        if not self.vectorizedArticles:
            raise ValueError("No articles have been analyzed yet! Run process_urls() first.")
        
        # Default priorities if not specified
        if student_priorities is None:
            student_priorities = {
                'growth_potential': 1.0,
                'recession_resistance': 1.0,
                'automation_resistance': 1.0,
                'skill_accessibility': 1.0,
                'cross_industry_collaboration': 1.0
            }
        
        # Normalize weights
        total_weight = sum(student_priorities.values())
        normalized_priorities = {k: v / total_weight for k, v in student_priorities.items()}
        
        logger.info(f"Analyzing recommendations for student interested in: {student_interests}")
        logger.info(f"Priority weights: {normalized_priorities}")
        
        # Step 1: Filter articles matching student interests
        df = pd.DataFrame(self.vectorizedArticles)
        
        if use_semantic_matching:
            # SEMANTIC MATCHING: Works great for abstract characteristics
            logger.info("Using semantic matching (better for characteristics like 'research focused')")
            
            # Create embedding for student's combined interests/characteristics
            student_profile = ' '.join(student_interests)
            student_embedding = self.analyzer.embedding_model.encode([student_profile], show_progress_bar=False)[0]
            
            # Calculate similarity scores for each article
            def calculate_similarity(row):
                """Calculate semantic similarity between student profile and article"""
                article_text = str(row.get('title', '')) + ' ' + str(row.get('content', ''))
                article_embedding = self.analyzer.embedding_model.encode([article_text], show_progress_bar=False)[0]
                
                # Cosine similarity
                similarity = np.dot(student_embedding, article_embedding) / (
                    np.linalg.norm(student_embedding) * np.linalg.norm(article_embedding)
                )
                return similarity
            
            df['similarity_score'] = df.apply(calculate_similarity, axis=1)
            df['matches_interest'] = df['similarity_score'] >= semantic_threshold
            
            logger.info(f"Similarity scores range: {df['similarity_score'].min():.3f} to {df['similarity_score'].max():.3f}")
            
            relevant_articles = df[df['matches_interest']].copy()
            
            # Boost composite score by similarity
            if len(relevant_articles) > 0:
                logger.info(f"Found {len(relevant_articles)} articles with similarity >= {semantic_threshold}")
            else:
                # Lower threshold if no matches
                logger.warning(f"No matches at threshold {semantic_threshold}, lowering to 0.2")
                semantic_threshold = 0.2
                df['matches_interest'] = df['similarity_score'] >= semantic_threshold
                relevant_articles = df[df['matches_interest']].copy()
        else:
            # KEYWORD MATCHING: Faster but only works for explicit terms
            logger.info("Using keyword matching (better for specific topics like 'AI')")
            interest_keywords = [interest.lower() for interest in student_interests]
            
            def matches_interests(row):
                """Check if article matches any student interest"""
                text = (str(row.get('title', '')) + ' ' + str(row.get('content', ''))).lower()
                return any(keyword in text for keyword in interest_keywords)
            
            df['matches_interest'] = df.apply(matches_interests, axis=1)
            df['similarity_score'] = df['matches_interest'].astype(float)  # Binary 0 or 1
            relevant_articles = df[df['matches_interest']].copy()
        
        if len(relevant_articles) == 0:
            logger.warning("No articles match student interests. Using top 20% of articles by growth potential.")
            cutoff = df['growth_potential'].quantile(0.8)
            relevant_articles = df[df['growth_potential'] >= cutoff].copy()
            relevant_articles['similarity_score'] = 0.5  # Neutral similarity
        
        logger.info(f"Found {len(relevant_articles)} articles matching student interests")
        
        # Step 2: Calculate weighted composite score for each article
        dimensions = ['growth_potential', 'recession_resistance', 'automation_resistance', 
                     'skill_accessibility', 'cross_industry_collaboration']
        
        relevant_articles['composite_score'] = 0
        for dim in dimensions:
            if dim in normalized_priorities:
                relevant_articles['composite_score'] += (
                    relevant_articles[dim] * normalized_priorities[dim]
                )
        
        # Boost composite score by similarity (articles more aligned with student get higher scores)
        if use_semantic_matching:
            relevant_articles['composite_score'] = (
                relevant_articles['composite_score'] * (1 + relevant_articles['similarity_score'])
            )
        
        # Step 3: Map articles to majors
        relevant_articles['related_majors'] = relevant_articles.apply(
            lambda row: self._extract_article_topics(row.to_dict()), 
            axis=1
        )
        
        # Step 4: Aggregate scores by major
        major_scores = {}
        major_article_counts = {}
        major_top_articles = {}
        
        for _, article in relevant_articles.iterrows():
            majors = article['related_majors']
            score = article['composite_score']
            title = article['title']
            
            for major in majors:
                if major not in major_scores:
                    major_scores[major] = []
                    major_article_counts[major] = 0
                    major_top_articles[major] = []
                
                major_scores[major].append(score)
                major_article_counts[major] += 1
                major_top_articles[major].append({
                    'title': title,
                    'score': score,
                    'growth': article['growth_potential'],
                    'recession': article['recession_resistance'],
                    'automation': article['automation_resistance']
                })
        
        # Step 5: Calculate average scores and create recommendations
        recommendations = []
        for major, scores in major_scores.items():
            avg_score = np.mean(scores)
            max_score = np.max(scores)
            
            # Sort articles by score and get top 3
            top_articles = sorted(major_top_articles[major], 
                                key=lambda x: x['score'], 
                                reverse=True)[:3]
            
            recommendations.append({
                'major': major.replace('_', ' ').title(),
                'average_score': avg_score,
                'max_score': max_score,
                'num_articles': major_article_counts[major],
                'top_article': top_articles[0]['title'] if top_articles else '',
                'avg_growth': np.mean([a['growth'] for a in top_articles]) if top_articles else 0,
                'avg_recession_resistance': np.mean([a['recession'] for a in top_articles]) if top_articles else 0,
                'avg_automation_resistance': np.mean([a['automation'] for a in top_articles]) if top_articles else 0,
            })
        
        # Convert to DataFrame and sort
        recommendations_df = pd.DataFrame(recommendations)
        
        if len(recommendations_df) == 0:
            logger.warning("No major recommendations could be generated.")
            return pd.DataFrame()
        
        recommendations_df = recommendations_df.sort_values('average_score', ascending=False)
        
        logger.info(f"\nTop {top_n} Major Recommendations:")
        for idx, row in recommendations_df.head(top_n).iterrows():
            logger.info(f"  {row['major']}: {row['average_score']:.2f} "
                       f"(based on {row['num_articles']} articles)")
        
        return recommendations_df.head(top_n)
    
    def explain_recommendation(self, major: str) -> Dict:
        """
        Get detailed explanation for why a major was recommended
        
        Args:
            major: The major to explain (e.g., 'Computer Science')
        
        Returns:
            Dictionary with detailed breakdown
        """
        df = pd.DataFrame(self.vectorizedArticles)
        major_key = major.lower().replace(' ', '_')
        
        # Find articles related to this major
        def is_related(row):
            topics = self._extract_article_topics(row.to_dict())
            return major_key in topics
        
        df['is_related'] = df.apply(is_related, axis=1)
        related_articles = df[df['is_related']]
        
        if len(related_articles) == 0:
            return {'error': f'No articles found related to {major}'}
        
        # Calculate statistics
        dimensions = ['growth_potential', 'recession_resistance', 'automation_resistance',
                     'skill_accessibility', 'cross_industry_collaboration']
        
        stats = {
            'major': major,
            'num_articles': len(related_articles),
            'dimension_scores': {
                dim: {
                    'mean': float(related_articles[dim].mean()),
                    'median': float(related_articles[dim].median()),
                    'min': float(related_articles[dim].min()),
                    'max': float(related_articles[dim].max())
                }
                for dim in dimensions
            },
            'top_articles': related_articles.nlargest(5, 'growth_potential')[
                ['title', 'growth_potential', 'recession_resistance', 'automation_resistance']
            ].to_dict('records')
        }
        
        return stats
    
    def diagnose_article_distribution(self) -> pd.DataFrame:
        """
        Diagnostic tool: See which majors your articles map to
        Use this to debug why you're getting limited recommendations
        """
        if not self.vectorizedArticles:
            logger.warning("No articles to diagnose!")
            return pd.DataFrame()
        
        major_distribution = {}
        article_details = []
        
        for article in self.vectorizedArticles:
            majors = self._extract_article_topics(article)
            
            article_details.append({
                'title': article['title'][:60] + '...',
                'majors': ', '.join([m.replace('_', ' ').title() for m in majors]) if majors else 'NONE',
                'num_majors': len(majors)
            })
            
            for major in majors:
                major_distribution[major] = major_distribution.get(major, 0) + 1
        
        # Print distribution
        print("\n" + "=" * 80)
        print("ARTICLE DISTRIBUTION BY MAJOR")
        print("=" * 80)
        
        if major_distribution:
            sorted_dist = sorted(major_distribution.items(), key=lambda x: x[1], reverse=True)
            for major, count in sorted_dist:
                percentage = (count / len(self.vectorizedArticles)) * 100
                print(f"{major.replace('_', ' ').title():30} {count:3} articles ({percentage:.1f}%)")
        else:
            print("⚠️  WARNING: NO ARTICLES MAPPED TO ANY MAJOR!")
            print("This means your keywords don't match your article content.")
        
        # Articles with no major mapping
        unmapped = [a for a in article_details if a['num_majors'] == 0]
        if unmapped:
            print(f"\n⚠️  {len(unmapped)} articles couldn't be mapped to any major:")
            for article in unmapped[:5]:
                print(f"  - {article['title']}")
        
        # Return detailed DataFrame
        details_df = pd.DataFrame(article_details)
        
        details_df.to_csv('labeled_fields.csv', index=False)
        
        return details_df



In [7]:
def create_training_csv():
    """
    Save your training data to a CSV file
    """
    
    # Same data structure as above
    training_data = {
        'text': [
            # Original domains / careers (37)
            "Emergency medicine and nursing require extensive hands-on training...",
            "Web development encompasses front-end and back-end programming...",
            "Food service and restaurant management involve food preparation...",
            "Artificial intelligence research focuses on developing novel...",
            "Plumbers install and repair water, drainage, and gas systems...",
            "LLMs are returning severely aggravating results when mistrained...",
            "Civil engineering focuses on designing and maintaining infrastructure such as roads...",
            "Cybersecurity analysts protect systems and networks from digital threats...",
            "Mechanical engineering applies physics and materials science to build machinery...",
            "Electrical line technicians install and maintain power transmission systems...",
            "Environmental science studies ecosystems, pollution mitigation, and sustainability...",
            "Biomedical engineering combines biology and engineering to develop medical devices...",
            "Pharmaceutical research develops and tests new medications and therapies...",
            "Construction project management oversees large-scale building projects...",
            "Data engineering focuses on building pipelines and managing large datasets...",
            "Robotics engineering designs automated systems and intelligent machines...",
            "Supply chain logistics coordinates production, transportation, and inventory...",
            "Agricultural science improves crop yields through biology and technology...",
            "Materials science studies the properties of metals, polymers, and composites...",
            "Urban planning designs land use and transportation for growing populations...",
            "Aerospace engineering develops aircraft, spacecraft, and propulsion systems...",
            "Renewable energy engineering focuses on solar, wind, and energy storage...",
            "Accounting and auditing manage financial records and regulatory compliance...",
            "Actuarial science applies statistics to assess financial risk...",
            "Economics research analyzes markets, incentives, and policy impacts...",
            "Technical writing translates complex concepts into accessible documentation...",
            "Human-computer interaction studies how people interact with technology...",
            "Industrial design creates functional and aesthetic consumer products...",
            "Chemistry research investigates chemical reactions and material synthesis...",
            "Quality assurance engineering tests systems for reliability and performance...",
            "Network engineering designs and maintains communication infrastructure...",
            "Geographic information systems analyze spatial and geospatial data...",
            "Manufacturing engineering optimizes production systems and processes...",
            "Education technology develops digital tools for learning environments...",
            "Public health analysis focuses on disease prevention and population health...",
            "Energy systems analysis evaluates energy production, efficiency, and policy...",
            "LLMs behaving badly: mistrained AI models quickly go off the rails...",
    
            "Collective intelligence for AI-assisted chemical synthesis using multiple specialized AI experts to generate executable experimental protocols...",
            "Nationwide real-world implementation of AI for cancer detection in population-based mammography screening...",
            "Multicenter evaluation of interpretable AI for coronary artery disease diagnosis from PET biomarkers...",
            "Generative AI-based low-dose digital subtraction angiography for intra-operative radiation dose reduction...",
            "AI-enabled virtual spatial proteomics from histopathology for interpretable biomarker discovery in lung cancer...",
            "Fair human-centric image dataset for ethical AI benchmarking...",
            "Increasing engagement with cognitive-behavioral therapy using generative AI in a randomized controlled trial...",
            "Holistic AI in medicine with multimodal data fusion and explainability...",
            "Integrative single-cell and spatial transcriptomics with explainable AI reveal lethal prognostic axis in prostate cancer...",
            "Sequence-based generative AI design of versatile tryptophan synthases...",
            "Chemistry-informed deep learning model for predicting stereoselectivity and absolute configuration in asymmetric hydrogenation...",
            "Artificial intelligence tools expand scientists’ impact but contract science’s focus...",
            "Discovery of highly fluorescent covalent organic frameworks through AI-assisted iterative experiment–learning cycles..."
        ],
    
        'growth_potential': [
            8,7,3,10,5,1,6,9,6,5,8,8,9,6,9,8,6,6,7,5,7,9,4,5,6,6,7,6,7,6,7,6,5,8,8,7,2,
            9,9,8,8,9,6,7,9,8,7,6,4,7
        ],
    
        'recession_resistance': [
            10,6,2,6,9,4,8,7,7,9,7,7,8,7,6,6,7,8,7,7,6,8,9,6,5,6,6,5,6,6,7,7,6,7,9,7,3,
            6,9,8,9,8,6,7,8,7,6,6,5,6
        ],
    
        'automation_resistance': [
            9,4,4,5,10,1,8,6,7,9,8,6,5,8,4,6,5,8,6,8,6,7,4,3,4,4,6,6,6,5,6,6,6,5,8,6,2,
            4,7,6,6,6,5,7,6,6,5,6,4,6
        ],
    
        'skill_accessibility': [
            2,8,7,1,6,10,4,5,4,5,6,3,2,5,6,3,6,6,3,5,2,5,7,3,4,8,6,6,4,5,6,6,5,7,4,4,9,
            2,3,3,2,2,6,4,2,2,3,3,6,2
        ],
    
        'cross_industry_collaboration': [
            8,5,3,10,4,9,7,6,6,4,8,9,9,6,8,9,7,6,8,6,8,9,4,5,7,6,8,7,6,6,8,7,6,8,8,8,6,
            9,8,7,7,8,6,7,8,7,7,6,5,8
        ]
    }
    df = pd.DataFrame(training_data)
    
    # Save to CSV
    df.to_csv('labeled_fields.csv', index=False)

create_training_csv()

In [5]:

if __name__ == "__main__":

    # Step 1: Train the analyzer on your labeled data
    print("=" * 80)
    print("STEP 1: Training the analyzer")
    print("=" * 80)
    
    # Load your labeled training data
    labeled_df = pd.read_csv('labeled_fields.csv')  # Your labeled dataset
    
    analyzer = MultiDimensionalAnalyzer()
    analyzer.train(
        texts=labeled_df['text'].tolist(),
        scores_df=labeled_df
    )
    


    # Step 2: Initialize the pipeline
    print("\n" + "=" * 80)
    print("STEP 2: Initializing pipeline")
    print("=" * 80)
    
    pipeline = ArticlePipeline(analyzer=analyzer)
    



    # Step 3: Scrape and analyze articles
    print("\n" + "=" * 80)
    print("STEP 3: Scraping and analyzing articles")
    print("=" * 80)
    
    # Example URLs (replace with your target sources)
    nature_scrapper = ArticleScraper(10, 3)

    articles = nature_scrapper.scrape_nature_smart(1030)
    
    print(len(articles))
    
    news_urls = [
        'https://www.nature.com/articles/d41586-026-00116-8',
        'https://www.nature.com/articles/d41586-025-04061-w',
        'https://www.nature.com/articles/s41586-025-09900-4',
        'https://www.nature.com/articles/d41586-026-00201-y',
        'https://www.nature.com/articles/d41586-026-00186-8',
        'https://www.nature.com/articles/d41586-026-00187-7',
        'https://www.nature.com/articles/d41586-026-00144-4',
        'https://www.nature.com/articles/d41586-026-00146-2',
        'https://www.nature.com/articles/d41586-026-00164-0',
        'https://www.nature.com/articles/d41586-026-00073-2',
        'https://www.nature.com/articles/d41586-025-04091-4',
        'https://www.nature.com/articles/d41586-025-04090-5',
        'https://www.nature.com/articles/d41586-025-04092-3',
        'https://www.nature.com/articles/d41586-025-04093-2',
        'https://www.nature.com/articles/d41586-025-04164-4',
        'https://www.nature.com/articles/s41586-026-10131-4',
        'https://www.nature.com/articles/s41586-025-09913-z',
        'https://www.nature.com/articles/d41586-026-00011-2',

    ]

    for i in range(len(articles)):
        news_urls.append(articles[i])
    
    print(f'number urls: {len(news_urls)}, starting to process.')
    # Process  articles
    results_df = pipeline.process_urls(news_urls, max_workers=5)
    print("\n URL's Processed!")
    
    # # Process arXiv papers
    # arxiv_ids = ['2401.12345', '2401.67890']  # Example arXiv IDs
    # arxiv_df = pipeline.process_arxiv_papers(arxiv_ids)
    

    pipeline.save_complete('articles.json', 'model.pkl')
 




2026-01-20 15:23:35,132 - INFO - Use pytorch device_name: cpu
2026-01-20 15:23:35,134 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


STEP 1: Training the analyzer


2026-01-20 15:23:37,621 - INFO - Creating embeddings for training data...
2026-01-20 15:23:37,893 - INFO - Training multi-output model...
2026-01-20 15:23:38,736 - INFO - Training complete!



STEP 2: Initializing pipeline

STEP 3: Scraping and analyzing articles
Step 1: Fetching subject categories from https://www.nature.com/subjects
Step 2: Found 103 subject categories
  1. https://www.nature.com/subjects/genetics
  2. https://www.nature.com/subjects/neurology
  3. https://www.nature.com/subjects/operational-research
  4. https://www.nature.com/subjects/developmental-biology
  5. https://www.nature.com/subjects/urology
  ... and 98 more

Step 3: Exploring subject - https://www.nature.com/subjects/genetics
  Checking pages...
  ✓ 10 articles found so far...

Step 3: Exploring subject - https://www.nature.com/subjects/neurology
  Checking pages...
  ✓ 20 articles found so far...

Step 3: Exploring subject - https://www.nature.com/subjects/operational-research
  Checking pages...
  ✓ 30 articles found so far...

Step 3: Exploring subject - https://www.nature.com/subjects/developmental-biology
  Checking pages...
  ✓ 40 articles found so far...
  ✓ 50 articles found so far...

2026-01-20 15:36:31,063 - INFO - Starting pipeline for 873 URLs...



COMPLETE: Found 855 articles from 103 subjects
855
number urls: 873, starting to process.


2026-01-20 15:53:22,174 - INFO - Successfully scraped 873/873 articles
2026-01-20 15:53:22,209 - INFO - Scoring 873 articles...
2026-01-20 15:54:18,081 - INFO - Pipeline complete! Total articles in vectorizedArticles: 873



 URL's Processed!
✓ Saved articles to articles.json
✓ Saved trained model to model.pkl


In [9]:
pipeline = ArticlePipeline(analyzer=None)   
pipeline.load_complete('articles.json', 'model.pkl')

labeled_df = pd.read_csv('labeled_fields.csv')  # Your labeled dataset

pipeline.analyzer.train(
    texts=labeled_df['text'].tolist(),
    scores_df=labeled_df
)


# Step 4: Access vectorizedArticles
print("\n" + "=" * 80)
print("STEP 4: Accessing vectorizedArticles")
print("=" * 80)

print(f"\nTotal articles processed: {len(pipeline.vectorizedArticles)}")
print("\nFirst 3 articles in vectorizedArticles:")
for i, article in enumerate(pipeline.vectorizedArticles[:3]):
    print(f"\n{i+1}. {article['title']}")
    print(f"   Growth: {article['growth_potential']:.1f}, "
        f"Recession: {article['recession_resistance']:.1f}, "
        f"Automation: {article['automation_resistance']:.1f}")
    

# Step 5: MAJOR RECOMMENDATION SYSTEM
print("\n" + "=" * 80)
print("STEP 5: DIAGNOSE ARTICLE DISTRIBUTION")
print("=" * 80)

# IMPORTANT: Run this first to see if your articles are mapping correctly
distribution = pipeline.diagnose_article_distribution()

print("\n" + "=" * 80)
print("STEP 6: MAJOR RECOMMENDATIONS FOR STUDENTS")
print("=" * 80)


# Example Student 1: Interested in AI and healthcare
print("\n--- Student 1: AI & Healthcare Enthusiast (Topic-based) ---")
student1_interests = ['artificial intelligence', 'machine learning', 'healthcare', 'medical']
student1_priorities = {
    'growth_potential': 1.0,          # Cares most about growth
    'recession_resistance': 0.8,      # Also values stability
    'automation_resistance': 0.5,     # Less concerned about automation
    'skill_accessibility': 0.3,       # Willing to do hard degrees
    'cross_industry_collaboration': 0.7
}   
recommendations1 = pipeline.recommend_major(
    student_interests=student1_interests,
    student_priorities=student1_priorities,
    top_n=5,
    use_semantic_matching=True  # Use semantic matching
)
print("\n" + recommendations1.to_string())


# Example Student 2: Characteristics-based (abstract traits)
print("\n\n--- Student 2: Research-Focused Abstract Thinker ---")
student2_interests = [
    'research focused', 
    'abstract thinking', 
    'theoretical work',
    'solving complex problems',
    'independent study'
]
student2_priorities = {
    'growth_potential': 1.0,
    'recession_resistance': 0.6,
    'automation_resistance': 0.8,      # Values non-automatable work
    'skill_accessibility': 0.4,        # Willing to pursue advanced degrees
    'cross_industry_collaboration': 0.5
}
recommendations2 = pipeline.recommend_major(
    student_interests=student2_interests,
    student_priorities=student2_priorities,
    top_n=5,
    use_semantic_matching=True,        # IMPORTANT: Use semantic for characteristics
)
print("\n" + recommendations2.to_string())


# Example Student 3: Human-centered characteristics
print("\n\n--- Student 3: People-Oriented Practical Problem Solver ---")
student3_interests = [
    'human element',
    'helping people',
    'hands-on work',
    'real-world impact',
    'working with communities',
    'practical applications'
]
student3_priorities = {
    'growth_potential': 0.7,
    'recession_resistance': 1.0,       # Wants job security
    'automation_resistance': 1.0,      # Wants human-centered work
    'skill_accessibility': 0.8,        # Prefers accessible entry
    'cross_industry_collaboration': 0.9
}
recommendations3 = pipeline.recommend_major(
    student_interests=student3_interests,
    student_priorities=student3_priorities,
    top_n=5,
    use_semantic_matching=True,
)
print("\n" + recommendations3.to_string())


# Example Student 4: Mix of topics and characteristics
print("\n\n--- Student 4: Creative Tech Innovator (Mixed Input) ---")
student4_interests = [
    'technology',                       # Topic
    'creative problem solving',         # Characteristic
    'innovation',                       # Characteristic
    'design',                          # Topic
    'user experience',                 # Topic
    'interdisciplinary thinking'        # Characteristic
]
student4_priorities = {
    'growth_potential': 1.0,
    'recession_resistance': 0.5,
    'automation_resistance': 0.7,
    'skill_accessibility': 0.9,
    'cross_industry_collaboration': 1.0
}
recommendations4 = pipeline.recommend_major(
    student_interests=student4_interests,
    student_priorities=student4_priorities,
    top_n=5,
    use_semantic_matching=True
)
print("\n" + recommendations4.to_string())



# Step 6: Detailed explanation for a specific major
print("\n" + "=" * 80)
print("STEP 6: DETAILED MAJOR EXPLANATION")
print("=" * 80)

if len(recommendations1) > 0:
    top_major = recommendations1.iloc[0]['major']
    explanation = pipeline.explain_recommendation(top_major)
    
    print(f"\nDetailed Analysis for {top_major}:")
    print(f"Based on {explanation['num_articles']} articles")
    print("\nDimension Scores:")
    for dim, scores in explanation['dimension_scores'].items():
        print(f"  {dim}: Mean={scores['mean']:.2f}, Range=[{scores['min']:.1f}, {scores['max']:.1f}]")
    
    print("\nTop Supporting Articles:")
    for i, article in enumerate(explanation['top_articles'][:3], 1):
        print(f"  {i}. {article['title'][:60]}...")
        print(f"     Growth: {article['growth_potential']:.1f}, "
                f"Recession: {article['recession_resistance']:.1f}")



# Step 7: Export results
print("\n" + "=" * 80)
print("STEP 7: EXPORT RESULTS")
print("=" * 80)

# Export article scores
pipeline.export_results('article_analysis_results.csv')

# Export recommendations
all_recommendations = pd.concat([recommendations1, recommendations2, recommendations3, recommendations4], 
                                ignore_index=True)
all_recommendations.to_csv('student_major_recommendations.csv', index=False)
print("Saved recommendations to 'student_major_recommendations.csv'")

# Access the vectorizedArticles variable directly
vectorizedArticles = pipeline.vectorizedArticles
print(f"\nvectorizedArticles contains {len(vectorizedArticles)} articles")

print("\n" + "=" * 80)
print("QUICK USAGE SUMMARY")
print("=" * 80)

2026-01-20 16:08:43,905 - INFO - Creating embeddings for training data...


✓ Loaded 873 articles
✓ Loaded trained model


2026-01-20 16:08:44,316 - INFO - Training multi-output model...
2026-01-20 16:08:45,334 - INFO - Training complete!



STEP 4: Accessing vectorizedArticles

Total articles processed: 873

First 3 articles in vectorizedArticles:

1. I’m going to halve my publication output. You should consider slow science, too
   Growth: 6.5, Recession: 6.7, Automation: 6.1

2. ‘Greed is the iron cage of our times’ — why nationalism is here to stay
   Growth: 6.7, Recession: 6.5, Automation: 5.4

3. Little red dots as young supermassive black holes in dense ionized cocoons - Nature
   Growth: 6.4, Recession: 6.1, Automation: 5.6

STEP 5: DIAGNOSE ARTICLE DISTRIBUTION


2026-01-20 16:19:39,148 - INFO - Analyzing recommendations for student interested in: ['artificial intelligence', 'machine learning', 'healthcare', 'medical']
2026-01-20 16:19:39,151 - INFO - Priority weights: {'growth_potential': 0.30303030303030304, 'recession_resistance': 0.24242424242424246, 'automation_resistance': 0.15151515151515152, 'skill_accessibility': 0.09090909090909091, 'cross_industry_collaboration': 0.21212121212121213}
2026-01-20 16:19:39,156 - INFO - Using semantic matching (better for characteristics like 'research focused')


...


                major  average_score  max_score  num_articles                                                                                                                                                                                 top_article  avg_growth  avg_recession_resistance  avg_automation_resistance
10          Marketing       7.642445   9.093863            24                                     Scaling to new heights: methodological insights for uplifting research on grassroots digital innovation - Humanities and Social Sciences Communications    7.150000                  6.856667                   6.210000
3           Sociology       7.525824   9.093863            93                                     Scaling to new heights: methodological insights for uplifting research on grassroots digital innovation - Humanities and Social Sciences Communications    6.903333                  6.546667                   6.203333
4          Psychology       7.466786   8.889389       

2026-01-20 16:30:57,931 - INFO - Similarity scores range: -0.133 to 0.423
2026-01-20 16:30:57,935 - INFO - Found 306 articles with similarity >= 0.1
2026-01-20 16:30:57,938 - INFO - Found 306 articles matching student interests
2026-01-20 16:34:38,356 - INFO - 
Top 5 Major Recommendations:
2026-01-20 16:34:38,358 - INFO -   Computer Science: 7.97 (based on 2 articles)
2026-01-20 16:34:38,359 - INFO -   Electrical Engineering: 7.69 (based on 4 articles)
2026-01-20 16:34:38,360 - INFO -   Marketing: 7.52 (based on 25 articles)
2026-01-20 16:34:38,361 - INFO -   Psychology: 7.46 (based on 54 articles)
2026-01-20 16:34:38,362 - INFO -   Sociology: 7.39 (based on 65 articles)



                     major  average_score  max_score  num_articles                                                                                                                                                       top_article  avg_growth  avg_recession_resistance  avg_automation_resistance
21        Computer Science       7.966849   8.557778             2                                                Companionship in code: AI’s role in the future of human connection - Humanities and Social Sciences Communications    7.130000                  6.605000                   5.615000
14  Electrical Engineering       7.687133   8.534303             4                                                 A systems engineering approach to enhance utilization of renewable energy - Nature Reviews Electrical Engineering    7.053333                  6.893333                   6.253333
8                Marketing       7.519819   9.282488            25           Scaling to new heights: methodological i

2026-01-20 16:42:08,054 - INFO - Exported 873 articles to article_analysis_results.csv



Detailed Analysis for Artificial Intelligence:
Based on 47 articles

Dimension Scores:
  growth_potential: Mean=6.55, Range=[4.5, 7.8]
  recession_resistance: Mean=6.26, Range=[5.0, 7.3]
  automation_resistance: Mean=5.23, Range=[3.6, 6.9]
  skill_accessibility: Mean=5.48, Range=[3.8, 7.8]
  cross_industry_collaboration: Mean=7.16, Range=[5.7, 7.8]

Top Supporting Articles:
  1. Validation of histopathology-based deep learning algorithms ...
     Growth: 7.8, Recession: 6.4
  2. A study on the coupling mechanism between the urban environm...
     Growth: 7.7, Recession: 6.5
  3. Applications of AI in urology - Nature Reviews Urology...
     Growth: 7.6, Recession: 6.5

STEP 7: EXPORT RESULTS
Saved recommendations to 'student_major_recommendations.csv'

vectorizedArticles contains 873 articles

QUICK USAGE SUMMARY


In [ ]:
custom_interests = [
    'research',
    'societal impact',
    'ground breaking discoveries',
    'fame',
    'money'
]

custom_priorities = {
    'growth_potential': 1.0,
    'recession_resistance': 1,
    'automation_resistance': 1,
    'skill_accessibility': 1,
    'cross_industry_collaboration': 1.0
}
    

custom_recc = pipeline.recommend_major(
    student_interests=custom_interests,
    student_priorities=custom_priorities,
    top_n=5,
    use_semantic_matching=True
)

2026-01-20 14:23:48,715 - INFO - Analyzing recommendations for student interested in: ['research', 'societal impact', 'ground breaking discoveries', 'fame', 'money']
2026-01-20 14:23:48,718 - INFO - Priority weights: {'growth_potential': 0.2, 'recession_resistance': 0.2, 'automation_resistance': 0.2, 'skill_accessibility': 0.2, 'cross_industry_collaboration': 0.2}
2026-01-20 14:23:48,722 - INFO - Using semantic matching (better for characteristics like 'research focused')
2026-01-20 14:23:52,721 - INFO - Similarity scores range: -0.077 to 0.455
2026-01-20 14:23:52,725 - INFO - Found 47 articles with similarity >= 0.1
2026-01-20 14:23:52,727 - INFO - Found 47 articles matching student interests


‘Greed is the iron cage of our times’ — why nationalism is here to stay 
Daily briefing: Symbols on ancient pottery could be earliest evidence of mathematics 
I’m going to halve my publication output. You should consider slow science, too 
A ‘time capsule’ for cells stores the secret experiences of their past 
Little red dots as young supermassive black holes in dense ionized cocoons - Nature 
Forget formalism: mathematics was built on infighting and emotional turmoil 
Can ‘toxic masculinity’ be measured? Scientists try to quantify controversial term 
Making progress on global health will need high-quality evidence 
What happens if fewer children get vaccinated? Japan holds lessons for US 
AI tools boost individual scientists but could limit research as a whole 
‘Shattered’: US scientists speak out about how Trump policies disrupted their careers 
Projections of 21st-century sea-level fall along coastal Greenland - Nature Communications 
Functional and structural insights into interact

2026-01-20 14:25:14,648 - INFO - 
Top 5 Major Recommendations:
2026-01-20 14:25:14,652 - INFO -   Biomedical Engineering: 8.53 (based on 1 articles)
2026-01-20 14:25:14,655 - INFO -   Neuroscience: 8.34 (based on 4 articles)
2026-01-20 14:25:14,657 - INFO -   Sociology: 8.04 (based on 6 articles)
2026-01-20 14:25:14,660 - INFO -   Political Science: 7.99 (based on 5 articles)
2026-01-20 14:25:14,662 - INFO -   Marketing: 7.93 (based on 3 articles)
